# NER Powered Semantic Search Using Pinecone v5.0.0

### Setup Environment

In [1]:
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

True

In [2]:
# init pinecone

from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.getenv("PINECONE"))


In [ ]:
# clean up pinecone index, after deleting all vectors if you run it again you will get error
index = pc.Index("medium-data")
index.delete(delete_all=True)
# delete index , dimension no longer useful
pc.delete_index("medium-data")

c:\Users\vasal\Study\VectorDB\.vecdb\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-10', 'x-cloud-trace-context': 'a04f517d83597e66965807419d6982b8', 'date': 'Sun, 28 Dec 2025 14:30:36 GMT', 'server': 'Google Frontend', 'Content-Length': '86', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"NOT_FOUND","message":"Resource medium-data not found"},"status":404}


In [4]:
# load libraries for NER 

from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline
import torch


### NER Engine

In [5]:
# init NER engine

model_id = 'dslim/bert-base-NER'

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(model_id)
device = torch.cuda.current_device() if torch.cuda.is_available() else 'cpu'

# nlp pipeline

nlp = pipeline('ner',
              model=model,
              tokenizer=tokenizer,
              aggregation_strategy= 'max',
              device= device) 
# nlp("Bill Gates is the founder of Microsoft")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


### Retriever

In [6]:
# load libraries for retriever

from sentence_transformers import SentenceTransformer


# https://huggingface.co/flax-sentence-embeddings/all_datasets_v3_mpnet-base
retriever = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v1")

In [7]:
retriever

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [8]:
# Create Pinecone Index
pc.create_index("medium-data", dimension= 768, metric="cosine",
                     spec=ServerlessSpec(cloud="aws", region="us-east-1"))


{
    "name": "medium-data",
    "metric": "cosine",
    "host": "medium-data-nm1v9fj.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "region": "us-east-1",
            "cloud": "aws",
            "read_capacity": {
                "mode": "OnDemand",
                "status": {
                    "state": "Ready",
                    "current_shards": null,
                    "current_replicas": null
                }
            }
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 768,
    "deletion_protection": "disabled",
    "tags": null,
    "_response_info": {
        "raw_headers": {
            "content-type": "application/json",
            "access-control-allow-origin": "*",
            "vary": "access-control-request-headers",
            "access-control-expose-headers": "*",
            "x-pinecone-api-version": "2025-10",
            "x-cloud-trace-contex

In [9]:
index= pc.Index("medium-data")

### Data Prep

In [13]:
from datasets import load_dataset

In [14]:
# Obtain Raw Data

df = load_dataset(
    "fabiochiu/medium-articles",
    data_files="medium_articles.csv",
    split="train"
).to_pandas()

df = df.dropna().sample(10000, random_state=45) # might take 30mins to 1hr

df['text_extended'] = df['title'] + '.' + df['text'].str[:1000]


c:\Users\vasal\Study\VectorDB\.vecdb\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vasal\.cache\huggingface\hub\datasets--fabiochiu--medium-articles. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular

In [ ]:
# Incase your internet is slow and couldn't make "dataset" works, you can download the file I uploaded as "medium_articles_10k.csv"
# Source of data: https://www.kaggle.com/code/fabiochiusano/medium-articles-simple-data-analysis?select=medium_articles.csv
# it is the same underlying data

# import pandas as pd
# df = pd.read_csv("data/medium_articles_10k.csv")
# df = df.dropna().sample(1000, random_state=45) # 
# df['text_extended'] = df['title'] + '.' + df['text'].str[:1000]

In [16]:
df

,title,text,url,authors,timestamp,tags,text_extended
189059,How do you move a WordPress website to another...,Photo by Moritz Mentges on Unsplash\n\nMoving ...,https://medium.com/@dyderik/how-do-you-move-a-...,['Richard Detering'],2021-11-13 05:42:44.009000+00:00,"['Web Hosting Services', 'Web Hosting', 'Trans...",How do you move a WordPress website to another...
96618,A Long December,In my quest to find ways of engaging with the ...,https://medium.com/@keenekomeskleen/a-long-dec...,['Matt Keene'],2020-12-16 19:47:55.820000+00:00,"['Society', 'Politics', 'Poverty', 'Pandemic',...",A Long December.In my quest to find ways of en...
46027,I Have Decided to Stop Being the Michael Scott...,"Writing is, for me, a beloved pastime. I’ve do...",https://pisancantos43.medium.com/i-have-decide...,['Anthony Aycock'],2019-05-12 20:04:59.371000+00:00,"['Teaching', 'Television', 'College', 'Writing...",I Have Decided to Stop Being the Michael Scott...
145790,EU fully committed to sustainable development,European Commission Vice-President Jyrki Katai...,https://medium.com/ecajournal/eu-fully-committ...,['European Court Of Auditors'],2019-07-24 13:15:13.130000+00:00,['Sustainable Development'],EU fully committed to sustainable development....
132859,HD ▷..! เรื่องเต็ม 【M-Thai ดาบพิฆาตอสูร เดอะมู...,TAG::\n\nดาบพิฆาตอสูร เดอะมูฟวี่ ศึกรถไฟสู่นิร...,https://medium.com/@bangetanjay405/hd-%E0%B9%8...,[],2020-12-12 14:33:34.766000+00:00,"['Thailand', 'Japan', 'Taiwan', 'Hong Kong', '...",HD ▷..! เรื่องเต็ม 【M-Thai ดาบพิฆาตอสูร เดอะมู...
...,...,...,...,...,...,...,...
55259,Mental Illness Issues,"Old records play, usually\n\nThey do not not f...",https://medium.com/written-tales/mental-illnes...,['David Kain'],2020-10-07 00:37:02.399000+00:00,"['Drug Addiction', 'Drinking', 'Written Tales'...","Mental Illness Issues.Old records play, usuall..."
31434,Bitkub.com officially received the DBD Registr...,"Launching recently on the 9th of May 2018, our...",https://medium.com/bitkub/bitkub-com-officiall...,[],2018-10-22 18:38:53.100000+00:00,"['News', 'Articles', 'Blockchain', 'Cryptocurr...",Bitkub.com officially received the DBD Registr...
29578,The impact of polling places on voting,Efforts to increase voter participation have l...,https://medium.com/mit-election-lab/the-impact...,['Mit Election Lab'],2019-07-11 15:53:05.269000+00:00,"['Esra2019', 'Politics', 'Voting']",The impact of polling places on voting.Efforts...
155202,Hunor Deak | FRUIT Magazine,Glasnost… Perestroika… Demokratizatsiya…\n\n…\...,https://medium.com/fruit-magazine-fiction/huno...,['Hunor Deak'],2020-10-18 05:56:00.735000+00:00,"['Soviet Union', 'Science Fiction', 'Space Exp...",Hunor Deak | FRUIT Magazine.Glasnost… Perestro...


In [ ]:
# len(nlp(df_batch)) # list of lst

### NER Helper Function

In [17]:
# helper function for extracting entities of a batch of texts

def extract_entities(list_of_text):
    entities = []
    for doc in list_of_text: 
        entities.append([item['word'] for item in nlp(doc)])
        # list of entities for 1 doc
    return entities

In [ ]:
# embedding

# len(retriever.encode(df_batch))
# len(retriever.encode(df_batch[0])) # try for one doc
# embedding for batch
# emb = retriever.encode(df_batch).tolist() # array to python list

### Batch Upsert

In [18]:
# upsert data

from tqdm.auto import tqdm

batch_size = 64

for i in range(0, len(df), batch_size):
    i_end = min(i+batch_size, len(df))
    # print(i, i_end) # starting and ending index of each batch
    
    # get a batch of data
    df_batch = df.iloc[i: i_end].copy()
    
    # embedding
    emb = retriever.encode(df_batch['text_extended'].tolist()
                          ).tolist() # array to python list
    
    # ner extraction
    entities = extract_entities(df_batch['text_extended'].tolist())
    
    # [[]] --> [set1, set2, ], remove duplicate entities    
    df_batch['named_entity'] = [list(set(entity)) for entity in entities] # one list per document
    
    # create meta data
    df_batch = df_batch.drop('text', axis=1)
    
    meta_data = df_batch.to_dict(orient='records') # pd.df to dictionary
    
    # create ids
    
    ids = [f"{idx}" for idx in range(i, i_end)] #
    
    # upsert
    
    vectors_to_upsert = list(zip(ids, emb, meta_data))  # nd array to python list
    
    _ = index.upsert(vectors= vectors_to_upsert)  

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
# index.describe_index_stats()

### Query data

In [19]:
query = "How to make a Wordpress website?"  # Natural Language

emb_qx = retriever.encode(query).tolist() # embedded query vector

ne = extract_entities([query])[0] # Named entity as a search filter

In [20]:
xc = index.query(vector=emb_qx, top_k= 5, include_metadata=True,
           filter = {"named_entity": {"$in" : ne}})

In [21]:
# you might not find any match if you are only upserting 1k data because of insufficient data there might not be good match, 
# try to load more data or tweak query based on data (glance over pinecone console and look for text_extended field in your vectors)
for result in xc['matches']:
    print(result['score'], " ", result['metadata']['named_entity'])

0.295511276   ['Both Sides of the Table', 'Wordpress']
0.290448219   ['Readymag', 'Industry', 'CMS', 'Wordpress']
0.227816597   ['Social Media Management', 'Zencart Cakephp', 'Prodigitaly', 'Squarespace', 'Codeignitor', 'Bigcommerce', 'Magento 2', 'Opencart', 'Wordpress', 'Laraval', 'Media', 'CMS', 'media', 'Joomla', 'IOS', 'Weebly', 'Woocomerce', 'Magento', 'Technologies and Platform Wordpress', 'Volusion', 'Drupal', 'Wix', 'Angular', 'Shopify', 'Banner']
0.135370269   ['Medium', 'Engine', 'YouTube', 'Cloud9', 'KeenGamer', 'HCS Raleigh Kickoff Major', 'Search', 'Wordpress']
0.0104198465   ['Godden & Baddeley', 'Abernethy', 'Wordpress']


In [22]:
query = "How to learn NLP?"  # Natural Language

emb_qx = retriever.encode(query).tolist() # embedded query vector

ne = extract_entities([query])[0] # Named entity as a search filter

xc = index.query(vector=emb_qx, top_k= 5, include_metadata=True,
           filter = {"named_entity": {"$in" : ne}  })

In [23]:
for result in xc['matches']:
    print(result['score'], " ", result['metadata']['named_entity'])

0.579979   ['Natural Language Processing', 'NLP']
0.497098953   ['NLP', 'ML']
0.47054103   ['Natural Language Processing', 'Sophia', 'Python', 'English', 'Alter', 'Machine Learning', 'Nadine', 'NLP', 'Artificial Intelligence', 'Deep Learning', 'Engineers', 'Analytics']
0.412491858   ['IE', 'NLP', 'AI', 'Information Extraction Pipeline', 'Knowledge']
0.401726753   ['Natural Language Processing', 'Xenon Stack Banks', 'Financial Services', 'NLP', 'AI']
